#  Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

---
###  What You Need to Implement
- Agent logic
- Conditional routing
- Tool integration
- Basic error handling

###  Bonus
- Improve routing
- Add logging
- Add more tools


## Calculator Tool
Evaluates a math expression via `eval`, wrapped in `try/except` so bad input returns `"Error in calculation"` instead of crashing.

In [1]:
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        return str(eval(expression))
    except Exception:
        return "Error in calculation"

##Keyword Extractor Tool
Splits text into words, keeps ones longer than 4 chars, dedupes, returns top 5 as keywords. Fails safe to `[]`.

In [2]:
def extract_keywords(text: str) -> list:
    """Extract keywords from text."""
    try:
        words = text.split()
        keywords = list(set([w.lower() for w in words if len(w) > 4]))
        return keywords[:5]
    except Exception:
        return []

##  Bonus: 3rd Tool + Logging Setup
Adds `text_stats` (word/char count tool), a `logger` for routing decisions, and `TRAJECTORY_LOG` to record every query→intent→response.

In [3]:
import logging

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")
logger = logging.getLogger("single_agent")

TRAJECTORY_LOG = []

def text_stats(text):
    """Return word count and character count for a piece of text."""
    try:
        words = text.split()
        return {"word_count": len(words), "char_count": len(text)}
    except Exception:
        return {"word_count": 0, "char_count": 0}

## 5 More Tools



Adds reverse text, uppercase converter, palindrome checker, temperature converter, and current date/time — so the agent now has 8 tools total.

In [4]:
# Reverse Text
def reverse_text(text):
    try:
        return text[::-1]
    except Exception:
        return ""


#  Uppercase Converter
def to_uppercase(text):
    try:
        return text.upper()
    except Exception:
        return ""


#  Palindrome Checker
def check_palindrome(text):
    try:
        skip_words = ["is", "a", "an", "the", "palindrome", "check", "if"]
        real_words = [w for w in text.lower().split() if w not in skip_words]
        cleaned = "".join(real_words)
        return cleaned == cleaned[::-1]
    except Exception:
        return False


#  Temperature Converter
import re

def convert_temperature(query):
    try:
        match = re.search(r"[-+]?\d*\.?\d+", query)
        if not match:
            return "No number found"
        value = float(match.group())

        q = query.lower()
        celsius_pos = q.find("celsius")
        fahrenheit_pos = q.find("fahrenheit")

        if celsius_pos == -1 and fahrenheit_pos == -1:
            return "Please mention celsius or fahrenheit"

        # the unit that appears first in the sentence is treated as the source unit
        if fahrenheit_pos != -1 and (celsius_pos == -1 or fahrenheit_pos < celsius_pos):
            return str(round((value - 32) * 5 / 9, 2)) + " C"
        else:
            return str(round((value * 9 / 5) + 32, 2)) + " F"
    except Exception:
        return "Error in conversion"


# Current Date & Time
from datetime import datetime

def get_current_datetime():
    try:
        return datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    except Exception:
        return ""

## Implement Agent Logic Below

 Use conditional routing:
- If query contains "calculate" → use calculator
- If query contains "keywords" → use keyword extractor
- Else → general response

##  Agent Function
Earlier it only matched exact words like "calculate" or "uppercase", so a differently worded question fell through to the general reply. Now each tool has a small list of words that can trigger it, and a bare math expression like "20 + 5" works even without the word "calculate".

In [5]:
import re

TRIGGERS = {
    "calculation": ["calculate", "compute", "solve"],
    "keywords": ["keyword", "keywords", "extract"],
    "stats": ["count", "stats", "statistics"],
    "reverse": ["reverse", "flip"],
    "uppercase": ["uppercase", "upper case", "capitalize", "caps"],
    "palindrome": ["palindrome"],
    "temperature": ["celsius", "fahrenheit", "temperature"],
    "datetime": ["date", "time", "today", "now"],
}

TOOLS = {
    "keywords": lambda q: extract_keywords(q),
    "stats": lambda q: text_stats(q),
    "reverse": lambda q: reverse_text(q),
    "uppercase": lambda q: to_uppercase(q),
    "palindrome": lambda q: check_palindrome(q),
    "temperature": lambda q: convert_temperature(q),
    "datetime": lambda q: get_current_datetime(),
}

def classify_intent(query_lower):
    if re.search(r"\d+\s*[\+\-\*/]\s*\d+", query_lower):
        return "calculation"
    for intent, words in TRIGGERS.items():
        if any(w in query_lower for w in words):
            return intent
    return "general"

def agent(query):
    query_lower = query.lower()
    intent = classify_intent(query_lower)
    logger.info("Query: " + query + " | Routed to: " + intent)

    try:
        if intent == "calculation":
            expression = re.sub(r"[a-zA-Z?]", "", query_lower).strip()
            result = calculator(expression)
            response = {"type": "error", "result": "Could not evaluate the expression."} if result == "Error in calculation" \
                else {"type": "calculation", "result": result}
        elif intent == "general":
            response = {"type": "general", "result": "I don't have a tool for this. Try calculate, keywords, stats, reverse, uppercase, palindrome, temperature, or date/time."}
        else:
            response = {"type": intent, "result": TOOLS[intent](query)}

    except Exception as e:
        logger.error("Agent failed: " + str(e))
        response = {"type": "error", "result": "Something went wrong: " + str(e)}

    TRAJECTORY_LOG.append({"query": query, "intent": intent, "response": response})
    return response

##  Expected Output Format

```
{
  "type": "calculation / keywords / general / error",
  "result": ...
}
```

## Test Cases
Runs the 3 sample queries (one per tool) through `agent()` and prints each response to confirm routing works.

In [6]:
queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?"
]

for q in queries:
    print("Query:", q)
    print("Response:", agent(q))
    print("-" * 50)

Query: Calculate 20 + 5
Response: {'type': 'calculation', 'result': '25'}
--------------------------------------------------
Query: Extract keywords from Artificial Intelligence is transforming industries
Response: {'type': 'keywords', 'result': ['intelligence', 'transforming', 'artificial', 'industries', 'keywords']}
--------------------------------------------------
Query: What is machine learning?
Response: {'type': 'general', 'result': "I don't have a tool for this. Try calculate, keywords, stats, reverse, uppercase, palindrome, temperature, or date/time."}
--------------------------------------------------


## Bonus: Stats Tool Test + Trajectory Log
Tests the bonus stats tool, then prints `TRAJECTORY_LOG` as a table showing every query, its routed intent, and its response.

In [7]:
print("Query: Show word count stats for this sentence")
print("Response:", agent("Show word count stats for this sentence"))
print("-" * 50)

import pandas as pd
pd.DataFrame(TRAJECTORY_LOG)

Query: Show word count stats for this sentence
Response: {'type': 'stats', 'result': {'word_count': 7, 'char_count': 39}}
--------------------------------------------------


,query,intent,response
0,Calculate 20 + 5,calculation,"{'type': 'calculation', 'result': '25'}"
1,Extract keywords from Artificial Intelligence ...,keywords,"{'type': 'keywords', 'result': ['intelligence'..."
2,What is machine learning?,general,"{'type': 'general', 'result': 'I don't have a ..."
3,Show word count stats for this sentence,stats,"{'type': 'stats', 'result': {'word_count': 7, ..."


##  Testing the 5 New Tools
Runs one query through each new tool to confirm all 8 tools work.

In [8]:
test_queries = [
    "Reverse this text",
    "Convert to uppercase please",
    "Is madam a palindrome",
    "Convert 100 celsius",
    "What is the current date and time"
]

for q in test_queries:
    print("Query:", q)
    print("Response:", agent(q))
    print("-" * 50)

Query: Reverse this text
Response: {'type': 'reverse', 'result': 'txet siht esreveR'}
--------------------------------------------------
Query: Convert to uppercase please
Response: {'type': 'uppercase', 'result': 'CONVERT TO UPPERCASE PLEASE'}
--------------------------------------------------
Query: Is madam a palindrome
Response: {'type': 'palindrome', 'result': True}
--------------------------------------------------
Query: Convert 100 celsius
Response: {'type': 'temperature', 'result': '212.0 F'}
--------------------------------------------------
Query: What is the current date and time
Response: {'type': 'datetime', 'result': '2026-07-12 14:08:43'}
--------------------------------------------------


## Cell 16 — Interactive Mode
Lets you type queries live and get agent responses until you type `exit`. Wrapped in `try/except` so it exits cleanly if no input is available (e.g. automated runs).

In [9]:
#  Interactive Mode

while True:
    try:
        user_input = input("Enter query (type 'exit' to stop): ")
    except Exception:
        print("(No interactive input available in this run — exiting loop.)")
        break
    if user_input.lower() == "exit":
        break
    print("Response:", agent(user_input))

Enter query (type 'exit' to stop): what is 67/3
Response: {'type': 'calculation', 'result': '22.333333333333332'}
Enter query (type 'exit' to stop): what is the time now
Response: {'type': 'datetime', 'result': '2026-07-12 14:09:14'}
Enter query (type 'exit' to stop): reverse Vikas
Response: {'type': 'reverse', 'result': 'sakiV esrever'}
Enter query (type 'exit' to stop): is naman a palindrome
Response: {'type': 'palindrome', 'result': True}
Enter query (type 'exit' to stop): capitalize this text Vikas
Response: {'type': 'uppercase', 'result': 'CAPITALIZE THIS TEXT VIKAS'}
Enter query (type 'exit' to stop): tell me the word count
Response: {'type': 'stats', 'result': {'word_count': 5, 'char_count': 22}}
Enter query (type 'exit' to stop): tell me the keywords
Response: {'type': 'keywords', 'result': ['keywords']}
Enter query (type 'exit' to stop): what is 34?2
Response: {'type': 'general', 'result': "I don't have a tool for this. Try calculate, keywords, stats, reverse, uppercase, palin

## Observation

While testing the agent, I noticed it correctly picks the right tool based on the words in the query. For example, if the query has "calculate" it goes to the calculator, if it has "uppercase" it converts the text, and so on. If none of the keywords match, it just gives a general reply instead of breaking.

I also tested some bad inputs, like an invalid math expression, and the agent handled it properly by returning an error message instead of crashing. The logs also showed every query along with which tool it was sent to, so it's easy to see what the agent is doing.

One thing I noticed is that the routing only works if the exact keyword is present in the query. So if someone asks in a different way, like "make this text loud" instead of "uppercase", the agent won't understand it.

## Conclusion

In this assignment, I built a single-agent smart assistant that can understand a query, decide which tool to use, run that tool, and give back a clean result. It handles math, keyword extraction, and general questions, and also has error handling so it doesn't crash on bad input.

For the bonus part, I added logging so every query gets recorded, and I added more tools than required (8 in total instead of just 2). Overall the agent works well for simple queries, and it's easy to add more tools later by just adding another condition in the routing logic.